# Spark Preparation
We check if we are in Google Colab.  If this is the case, install all necessary packages.

To run spark in Colab, we need to first install all the dependencies in Colab environment i.e. Apache Spark 3.3.2 with hadoop 3.3, Java 8 and Findspark to locate the spark in the system. The tools installation can be carried out inside the Jupyter Notebook of the Colab.
Learn more from [A Must-Read Guide on How to Work with PySpark on Google Colab for Data Scientists!](https://www.analyticsvidhya.com/blog/2020/11/a-must-read-guide-on-how-to-work-with-pyspark-on-google-colab-for-data-scientists/)

In [1]:
try:
  import google.colab
  IN_COLAB = True
except:
  IN_COLAB = False

In [2]:
if IN_COLAB:
    !apt-get install openjdk-8-jdk-headless -qq > /dev/null
    !wget -q https://dlcdn.apache.org/spark/spark-3.3.2/spark-3.3.2-bin-hadoop3.tgz
    !tar xf spark-3.3.2-bin-hadoop3.tgz
    !mv spark-3.3.2-bin-hadoop3 spark
    !pip install -q findspark
    import os
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
    os.environ["SPARK_HOME"] = "/content/spark"

# Start a Local Cluster

In [3]:
from functools import reduce
from pyspark.sql.functions import (col, trim, lower, regexp_replace, sum, udf, to_timestamp,split, datediff, substring, 
    current_timestamp, when, datediff, try_to_timestamp, to_date)
from pythainlp import word_tokenize
from pyspark.sql.types import ArrayType, StringType
from pythainlp.corpus import thai_stopwords



In [4]:
spark_url = 'local'
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType

spark = SparkSession.builder \
    .appName("TraffyFondueDataCleaning") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

In [5]:
sc = spark.sparkContext

file_path = 'bangkok_traffy.csv'

## schema

In [6]:
traffy_schema = StructType([
    # ตัวระบุเฉพาะ
    StructField("ticket_id", StringType(), True),
    
    # ข้อมูลปัญหาและการจัดการ
    StructField("type", StringType(), True),         # หมวดหมู่ปัญหา
    StructField("organization", StringType(), True), # หน่วยงานที่รับผิดชอบ
    StructField("comment", StringType(), True),      # ข้อความร้องเรียน (สำคัญสำหรับ LLM)
    StructField("photo", StringType(), True),
    StructField("photo_after", StringType(), True),
    
    # ข้อมูลพิกัดและตำแหน่ง
    StructField("coords", StringType(), True),       # พิกัด Lat/Long (เก็บเป็น String ก่อนแล้วค่อย Parse)
    StructField("address", StringType(), True),
    StructField("subdistrict", StringType(), True),
    StructField("district", StringType(), True),
    StructField("province", StringType(), True),
    
    # ข้อมูลเวลาและสถานะ
    StructField("timestamp", StringType(), True),    # วันที่สร้าง (เก็บเป็น String ก่อนแล้วค่อย Cast เป็น Timestamp)
    StructField("state", StringType(), True),        # สถานะปัจจุบัน (ใช้ในการ Filter Active Issues)
    
    # ข้อมูลการตอบรับและกิจกรรม
    StructField("star", FloatType(), True),          # เรทติ้ง 0-5
    StructField("count_reopen", IntegerType(), True), # จำนวนครั้งที่เปิดซ้ำ
    StructField("last_activity", StringType(), True)  # วันที่กิจกรรมล่าสุด (เก็บเป็น String ก่อน)
])

In [7]:
df_traffy = spark.read.csv(
    file_path,
    header=True,
    schema=traffy_schema,
    multiLine=True, # สำคัญ: หาก 'comment' หรือ 'address' มีหลายบรรทัด
    escape='"' # สำคัญ: หากมีเครื่องหมายคำพูดในข้อความ
)

In [8]:
df_traffy.select("last_activity").show(10, False)


+-----------------------------+
|last_activity                |
+-----------------------------+
|2022-06-04 15:34:14.609206+00|
|2022-06-21 08:21:09.532782+00|
|2022-06-06 01:17:12.272904+00|
|2022-09-08 08:35:43.784519+00|
|2022-08-12 07:18:44.884945+00|
|2023-03-14 12:09:14.947437+00|
|2023-05-17 06:11:32.463984+00|
|2024-11-26 04:17:39.760344+00|
|2022-06-24 06:32:34.671236+00|
|2022-06-20 13:12:04.99444+00 |
+-----------------------------+
only showing top 10 rows


In [9]:
print(f"จำนวนแถวเริ่มต้น: {df_traffy.count()}")
df_traffy.printSchema()

จำนวนแถวเริ่มต้น: 787026
root
 |-- ticket_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- organization: string (nullable = true)
 |-- comment: string (nullable = true)
 |-- photo: string (nullable = true)
 |-- photo_after: string (nullable = true)
 |-- coords: string (nullable = true)
 |-- address: string (nullable = true)
 |-- subdistrict: string (nullable = true)
 |-- district: string (nullable = true)
 |-- province: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- state: string (nullable = true)
 |-- star: float (nullable = true)
 |-- count_reopen: integer (nullable = true)
 |-- last_activity: string (nullable = true)



In [10]:


# ลิสต์หมวดหมู่หลักที่ส่งผลต่อมูลค่าอสังหาฯ และความน่าอยู่
livability_types = [
    "ถนน",
    "ทางเท้า",
    "ความปลอดภัย",
    "แสงสว่าง",
    "ความสะอาด",
    "กีดขวาง",
    "ท่อระบายน้ำ",
    "น้ำท่วม",
    "ต้นไม้",
    "PM2.5",
    "จราจร",
    "สะพาน"
]

# กรองข้อมูลตาม 'type' (หมวดหมู่ปัญหา)
# ใช้วิธี 'isin' ที่ตรงไปตรงมาที่สุด
df_filtered_type = df_traffy.filter(
    reduce(lambda a, b: a | b, [col("type").contains(t) for t in livability_types])
)






In [11]:
df_final_spatial = df_filtered_type.withColumn(
    "lat", 
    trim(split(col("coords"), ",").getItem(0)).cast("float")
).withColumn(
    "lon", 
    trim(split(col("coords"), ",").getItem(1)).cast("float")
).filter(col("lat").isNotNull() & col("lon").isNotNull()) # กรองแถวที่แปลงพิกัดไม่ได้



print(f"DataFrame สุดท้ายพร้อมสำหรับ LLM/Join: {df_final_spatial.count()} แถว")
df_final_spatial.select("lat", "lon").show(5, truncate=False)


DataFrame สุดท้ายพร้อมสำหรับ LLM/Join: 596793 แถว
+---------+--------+
|lat      |lon     |
+---------+--------+
|100.53084|13.81865|
|100.66709|13.67891|
|100.52649|13.7206 |
|100.53099|13.81853|
|100.59165|13.8228 |
+---------+--------+
only showing top 5 rows


In [12]:
df_trim_string = df_final_spatial.withColumn(
    "timestamp_str", 
    substring(col("timestamp"), 1, 19) # เริ่มจาก index 1, เอา 19 ตัวอักษร
).withColumn(
    "last_activity_str", 
    substring(col("last_activity"), 1, 19) # ทำเหมือนกันกับ last_activity
)

# Format ที่ใช้หลังตัด:
TIMESTAMP_FORMAT_SIMPLE = "yyyy-MM-dd HH:mm:ss"

# 2. แปลง String เป็น Timestamp (TimestampType)
df_time_prep = df_trim_string.withColumn(
    "timestamp_dt", 
    to_timestamp(col("timestamp_str"), TIMESTAMP_FORMAT_SIMPLE)
).withColumn(
    "last_activity_dt", 
    to_timestamp(col("last_activity_str"), TIMESTAMP_FORMAT_SIMPLE)
)

# กรองแถวที่แปลง timestamp ไม่ได้ (Timestamp/last_activity เป็น NULL หลังแปลง)
df_time_prep = df_time_prep.filter(
    col("timestamp_dt").isNotNull() 
)

# 3. แปลงเป็น Date และคำนวณ DaysToFix
df_time_prep = df_time_prep.withColumn("timestamp_date", to_date(col("timestamp_dt")))
df_time_prep = df_time_prep.withColumn("last_activity_date", to_date(col("last_activity_dt")))

df_final_ready = df_time_prep.withColumn(
    "DaysToFix",
    when(
        # ถ้า state = 'เสร็จสิ้น'
        col("state") == "เสร็จสิ้น",
        datediff(col("last_activity_date"), col("timestamp_date"))
    ).otherwise(
        # ถ้า state = 'กำลังดำเนินการ' หรือ 'รอรับเรื่อง'
        datediff(to_date(current_timestamp()), col("timestamp_date"))
    )
)

# ตรวจสอบผลลัพธ์
df_final_ready.select(
    "ticket_id", "state", "lat", "lon", 
    "timestamp_dt", "last_activity_dt", "DaysToFix"
).show(5, truncate=False)

+-----------+---------+---------+--------+-------------------+-------------------+---------+
|ticket_id  |state    |lat      |lon     |timestamp_dt       |last_activity_dt   |DaysToFix|
+-----------+---------+---------+--------+-------------------+-------------------+---------+
|2021-FYJTFP|เสร็จสิ้น|100.53084|13.81865|2021-09-03 12:51:09|2022-06-04 15:34:14|274      |
|2021-CGPMUN|เสร็จสิ้น|100.66709|13.67891|2021-09-19 14:56:08|2022-06-21 08:21:09|275      |
|2021-7XATFA|เสร็จสิ้น|100.52649|13.7206 |2021-09-26 05:03:52|2022-06-06 01:17:12|253      |
|2021-9U2NJT|เสร็จสิ้น|100.53099|13.81853|2021-10-14 10:45:27|2022-09-08 08:35:43|329      |
|2021-DVEWYM|เสร็จสิ้น|100.59165|13.8228 |2021-12-09 12:29:08|2022-08-12 07:18:44|246      |
+-----------+---------+---------+--------+-------------------+-------------------+---------+
only showing top 5 rows


In [13]:

df_clean = (
    df_final_ready
    .withColumn("comment_clean", trim(col("comment")))
    .withColumn("comment_clean", lower(col("comment_clean")))
    .withColumn("comment_clean", regexp_replace(col("comment_clean"), "[\n\r\t]", " "))
    .withColumn("comment_clean", regexp_replace(col("comment_clean"), "[^ก-๙a-z0-9/. ]", ""))
    .withColumn("comment_clean", regexp_replace(col("comment_clean"), " +", " "))
)
df_clean.select("comment_clean").show(20, truncate=False)

+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|comment_clean                                                                                                                                                                                                                                                                       |
+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|ขยะเยอะ                                                                                                                                                           

In [14]:
df_clean = df_clean.filter(col("comment_clean").isNotNull())
print(df_clean.count())

591838


In [15]:

# @udf(ArrayType(StringType()))
# def thai_tokenize(text):
#     if text and text.strip():
#         # ใช้ Engine 'newmm' และลบช่องว่างออกจาก tokens
#         return word_tokenize(text, engine='newmm', keep_whitespace=False)
#     else:
#         return []

# # UDF สำหรับการลบ Stopwords
# # โหลด Stopwords List แค่ครั้งเดียว
# STOPWORDS = set(thai_stopwords())
# @udf(ArrayType(StringType()))
# def remove_stopwords(tokens):
#     if tokens is not None:
#         return [word for word in tokens if word not in STOPWORDS and word != '']
#     else:
#         return []


# # A. สร้างคอลัมน์ tokens จาก 'comment_clean'
# df_tokenized = df_clean.withColumn(
#     "tokens", 
#     thai_tokenize(col("comment_clean")) # แก้ไขเป็น 'comment_clean' แล้ว
# )

# # B. สร้างคอลัมน์ final_tokens (สำหรับ LLM Sentiment)
# df_final_llm = df_tokenized.withColumn(
#     "final_tokens", 
#     remove_stopwords(col("tokens"))
# )

# df_final_llm.select("comment_clean", "final_tokens").show(5, truncate=False)

In [16]:

df_clean.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_clean.columns
]).show()


+---------+----+------------+-------+-----+-----------+------+-------+-----------+--------+--------+---------+-----+------+------------+-------------+---+---+-------------+-----------------+------------+----------------+--------------+------------------+---------+-------------+
|ticket_id|type|organization|comment|photo|photo_after|coords|address|subdistrict|district|province|timestamp|state|  star|count_reopen|last_activity|lat|lon|timestamp_str|last_activity_str|timestamp_dt|last_activity_dt|timestamp_date|last_activity_date|DaysToFix|comment_clean|
+---------+----+------------+-------+-----+-----------+------+-------+-----------+--------+--------+---------+-----+------+------------+-------------+---+---+-------------+-----------------+------------+----------------+--------------+------------------+---------+-------------+
|        0|   0|         428|      0|   74|     103922|     0|      0|        491|     489|     162|        0|    0|390222|           0|            0|  0|  0|     

In [17]:
df_map_sample_pandas = df_final_ready.sample(
    fraction=10000 / df_final_ready.count(), 
    seed=42
).toPandas()
df_map_sample_pandas.describe()

,star,count_reopen,lat,lon,timestamp_dt,last_activity_dt,DaysToFix
count,3477.000000,10082.000000,10082.000000,10082.000000,10082,10082,10082.000000
mean,3.912856,0.112081,100.512154,13.757599,2023-10-05 03:34:57.170204416,2023-12-17 04:12:20.817198848,171.091648
min,1.000000,0.000000,0.000000,0.000000,2022-01-17 03:01:52,2022-06-04 00:23:55,0.000000
25%,3.000000,0.000000,100.505032,13.717168,2023-02-21 07:50:16,2023-05-30 06:52:20.500000,3.000000
50%,5.000000,0.000000,100.558598,13.752440,2023-10-23 09:09:38,2024-01-19 02:57:04.500000,19.000000
75%,5.000000,0.000000,100.624447,13.799115,2024-06-09 14:07:28.750000128,2024-08-06 09:50:10.249999872,255.000000
max,5.000000,18.000000,100.933083,13.951480,2025-01-16 02:53:34,2025-01-16 02:58:02,1269.000000
std,1.422185,0.647870,2.235914,0.204009,NaN,NaN,273.535181


In [18]:
# -----------------------------
# 1️⃣ Import libraries
# -----------------------------
from pyspark.sql.functions import col, split
import pandas as pd
import numpy as np
import folium
from folium.plugins import HeatMap

# -----------------------------
# 2️⃣ แยก lon / lat จาก coords string
# -----------------------------
# สมมติ coords เป็น "lon,lat"
df_final_ready = df_final_ready.withColumn("lon", split(col("coords"), ",").getItem(0).cast("double")) \
                               .withColumn("lat", split(col("coords"), ",").getItem(1).cast("double"))

# -----------------------------
# 3️⃣ Sample data (10,000 rows) และแปลงเป็น Pandas
# -----------------------------
n_sample = 10000
n_total = df_final_ready.count()
df_sample_pandas = df_final_ready.sample(fraction=n_sample / n_total, seed=42).toPandas()

# -----------------------------
# 4️⃣ เตรียม weight (DaysToFix) แบบ log scale
# -----------------------------
df_sample_pandas['weight'] = np.log1p(df_sample_pandas['DaysToFix'])

# -----------------------------
# 5️⃣ เตรียมข้อมูลสำหรับ HeatMap
# -----------------------------
data_heatmap = df_sample_pandas[['lat', 'lon', 'weight']].values.tolist()

# -----------------------------
# 6️⃣ สร้าง Folium Map
# -----------------------------
center_lat = 13.737
center_lon = 100.528

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=11,
    tiles="cartodbpositron"
)

# -----------------------------
# 7️⃣ เพิ่ม HeatMap Layer
# -----------------------------
HeatMap(
    data_heatmap,
    radius=10,            # ขนาดจุด
    blur=15,              # ความฟุ้ง
    max_val=df_sample_pandas['weight'].max()
).add_to(m)

# -----------------------------
# 8️⃣ แสดงผล (Jupyter Notebook) / บันทึกเป็น HTML
# -----------------------------
m  # Interactive map in notebook
m.save("daystofix_heatmap.html")  # บันทึกเป็น HTML


C:\Users\asdwer\AppData\Local\Temp\ipykernel_16844\606478017.py:49: UserWarning: The `max_val` parameter is no longer necessary. The largest intensity is calculated automatically.
  HeatMap(
